# Dubby 🐨 on Google Colab

**YouTube dubbing studio · 🇺🇸 🇪🇬 🇸🇦 🇪🇸 🇫🇷 🇮🇹 🇮🇳 🇨🇳 🇯🇵**

Run the cells **one by one, top to bottom**. At the end you get a link to the full Dubby web studio running on Colab's GPU.

| Step | What happens |
|---|---|
| 0️⃣ | Helpers: every install step **stops with a clear error** if something fails |
| 1️⃣ | GPU check |
| 2️⃣ | Get the Dubby code |
| 3️⃣ | Node.js 22 (UI build + YouTube JS challenges) |
| 4️⃣ | **Core** engines in an isolated Python 3.12 venv: WhisperX, Cohere Transcribe/CohereX, Metro-ASR, NLLB, Hunyuan-MT, oddadmix translators, OmniVoice, VoiceTut, Lahgtna |
| 5️⃣ | *(optional)* **Qwen** (Qwen3-ASR, QwenCleo, Qwen3-TTS), **NeMo** (Parakeet), **Indic** (IndicTrans2, IndicF5 for Hindi) |
| 6️⃣ | Hugging Face token (gated: Cohere, IndicF5, IndicTrans2) |
| 7️⃣ | Build the UI, check engines, launch the studio |

> 💡 **Runtime → Change runtime type → T4 GPU** (L4/A100 for Hunyuan-MT-7B) before starting.
> Dubby installs into `/content/envs/*` venvs, so Colab's preinstalled Python 3.13 packages never conflict with it.

## 0️⃣ Helpers

In [ ]:
import os, subprocess, sys, time
ENVS = "/content/envs"
# Not "/content/dubby": a folder named like the package would shadow `import dubby`.
REPO_DIR = "/content/dubby-studio"
# Colab exports UV_SYSTEM_PYTHON=1, which makes uv warn inside our venvs.
os.environ.pop("UV_SYSTEM_PYTHON", None)
os.environ["PYTHONWARNINGS"] = "ignore::SyntaxWarning"

def run(cmd, cwd="/tmp", show=True):
    """Run a shell command, stream its output and raise if it fails.

    Commands run from /tmp by default so a stray `dubby` folder in /content can never
    shadow the installed package.
    """
    print(f"\n$ {cmd}", flush=True)
    proc = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in proc.stdout:
        tail = (tail + [line])[-60:]
        if show:
            print(line, end="", flush=True)
    if proc.wait() != 0:
        if not show:
            print("".join(tail))
        raise RuntimeError(f"❌ Command failed (exit {proc.returncode}): {cmd}")

def venv_python(name):
    return f"{ENVS}/{name}/bin/python"

def add_to_path(path):
    if path not in os.environ["PATH"].split(":"):
        os.environ["PATH"] = f"{path}:{os.environ['PATH']}"

print("✅ helpers ready")

## 1️⃣ Check the GPU

In [ ]:
run("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv || echo 'No GPU — switch the runtime to a GPU'")
print(sys.version)

## 2️⃣ Get the Dubby code

In [ ]:
#@title Clone the repository { display-mode: "form" }
REPO_URL = "https://github.com/MohammedAly22/Dubby"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
if os.path.isdir(f"{REPO_DIR}/.git"):
    run(f"git -C {REPO_DIR} pull --ff-only")
else:
    run(f"git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO_DIR}")
print("✅ code ready at", REPO_DIR)

## 3️⃣ Node.js 22 + system tools

In [ ]:
#@title Node.js, ffmpeg and sox { display-mode: "form" }
NODE_VERSION = "22.20.0"  #@param {type:"string"}
node_dir = f"/usr/local/lib/node-v{NODE_VERSION}-linux-x64"
if not os.path.isdir(node_dir):
    run(f"curl -fsSL https://nodejs.org/dist/v{NODE_VERSION}/node-v{NODE_VERSION}-linux-x64.tar.xz | tar -xJ -C /usr/local/lib")
add_to_path(f"{node_dir}/bin")
run("apt-get -qq update && apt-get -qq install -y ffmpeg sox libsox-dev > /dev/null", show=False)
run("node --version && npm --version && ffmpeg -version | head -n 1")
print("✅ system tools ready")

## 4️⃣ Core engines (≈ 6–8 min)

Installed with [`uv`](https://github.com/astral-sh/uv) into an isolated **Python 3.12** venv at `/content/envs/dubby`, so it never fights Colab's own packages.

In [ ]:
#@title Core install { display-mode: "form" }
INSTALL_DEMUCS = True  #@param {type:"boolean"}
TORCH = "torch==2.8.0 torchaudio==2.8.0 torchvision==0.23.0"
CU = "--extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match"

run("pip install -q -U uv", show=False)
if not os.path.exists(venv_python("dubby")):
    run(f"uv venv --python 3.12 --seed {ENVS}/dubby")
PY = venv_python("dubby")
run(f"uv pip install --python {PY} {TORCH} {CU}")
run(f"uv pip install --python {PY} -e '{REPO_DIR}[core]' {CU}")
if INSTALL_DEMUCS:
    run(f"uv pip install --python {PY} demucs {CU}")

add_to_path(f"{ENVS}/dubby/bin")
run(f"{PY} -c \"import dubby, whisperx, omnivoice, voicetut_tts, torch; print('dubby', dubby.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())\"")
run("which dubby && dubby --version")
# YouTube token helper: lets downloads pass YouTube's bot check without cookies or sign-in
run("dubby youtube-helper --check")
print("✅ core engines installed")

## 5️⃣ Optional engine families

Each family gets its own venv, which Dubby detects automatically:

* **Qwen** (`/content/envs/dubby-qwen`): Qwen3-ASR, QwenCleo-ASR (Egyptian) and Qwen3-TTS. `qwen-asr` pins transformers 4.57.
* **NeMo** (`/content/envs/dubby-nemo`): NVIDIA Parakeet TDT (en/es/fr/it).
* **Indic** (`/content/envs/dubby-indic`): IndicTrans2 (English→Hindi) and IndicF5 (Hindi TTS). Both need transformers < 4.50.

In [ ]:
#@title Qwen · NeMo · Indic families { display-mode: "form" }
INSTALL_QWEN = True  #@param {type:"boolean"}
INSTALL_NEMO = False  #@param {type:"boolean"}
INSTALL_INDIC = False  #@param {type:"boolean"}
TORCH = "torch==2.8.0 torchaudio==2.8.0"
CU = "--extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match"

def family(name, extra, post=()):
    env = f"{ENVS}/dubby-{name}"
    if not os.path.exists(f"{env}/bin/python"):
        run(f"uv venv --python 3.12 --seed {env}")
    py = f"{env}/bin/python"
    run(f"uv pip install --python {py} {TORCH} {CU}")
    run(f"uv pip install --python {py} -e '{REPO_DIR}[{extra}]' {CU}")
    for cmd in post:
        run(cmd.format(py=py, CU=CU))
    run(f"{py} -m dubby.workers.doctor --family {name}")
    print(f"✅ {name} family ready")

if INSTALL_QWEN:
    family("qwen", "qwen", post=[
        "uv pip install --python {py} qwencleo-asr --no-deps",
        "uv pip install --python {py} qwen-tts --no-deps",
        "uv pip install --python {py} onnxruntime einops sox {CU}",
    ])
if INSTALL_NEMO:
    family("nemo", "nemo")
if INSTALL_INDIC:
    family("indic", "indic", post=[
        "uv pip install --python {py} 'git+https://github.com/ai4bharat/IndicF5.git' 'transformers<4.50' {CU}",
    ])

## 6️⃣ Hugging Face token

Accept the terms on the gated models you plan to use, then paste a *read* token (or store it as a Colab secret named `HF_TOKEN`):
[Cohere Transcribe](https://huggingface.co/CohereLabs/cohere-transcribe-03-2026) ·
[Cohere Transcribe Arabic](https://huggingface.co/CohereLabs/cohere-transcribe-arabic-07-2026) ·
[IndicF5](https://huggingface.co/ai4bharat/IndicF5) ·
[IndicTrans2](https://huggingface.co/ai4bharat/indictrans2-en-indic-1B)

In [ ]:
#@title Token { display-mode: "form" }
HF_TOKEN = ""  #@param {type:"string"}
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN") or ""
    except Exception:
        pass
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    run(f"{venv_python('dubby')} -c \"from huggingface_hub import login; login(token='{HF_TOKEN}', add_to_git_credential=False); print('logged in')\"", show=False)
    print("✅ Hugging Face token set")
else:
    print("ℹ️ No token: gated engines (Cohere, IndicF5, IndicTrans2) will be unavailable; everything else works.")

## 7️⃣ Build the UI, check engines & languages

In [ ]:
add_to_path(f"{ENVS}/dubby/bin")
os.environ.update({"FORCE_COLOR": "1", "COLUMNS": "120", "DUBBY_HOME": "/content/Dubby", "MPLBACKEND": "Agg"})
run("dubby build-ui", cwd=REPO_DIR)
run("dubby doctor")
run("dubby languages")

## 🚀 Launch the studio

The studio terminal (stage banners, live transcripts, translations, clip timings) goes to `/content/dubby.log`; the next cell shows it.
Pick **Colab proxy** (quickest) or **Cloudflare tunnel** (a public `trycloudflare.com` link that also works on another device).

In [ ]:
#@title Start Dubby { display-mode: "form" }
TUNNEL = "colab-proxy"  #@param ["colab-proxy", "cloudflare"]
EXPORT_TO_DRIVE = False  #@param {type:"boolean"}
import json, re, urllib.request

DUBBY = f"{ENVS}/dubby/bin/dubby"
if not os.path.exists(DUBBY):
    raise RuntimeError("Dubby is not installed — run step 4 first.")
add_to_path(f"{ENVS}/dubby/bin")
os.environ.update({"DUBBY_HOME": "/content/Dubby", "FORCE_COLOR": "1", "COLUMNS": "120", "PYTHONUNBUFFERED": "1"})
if EXPORT_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("/content/Dubby", exist_ok=True)
    settings_path = "/content/Dubby/settings.json"
    data = json.load(open(settings_path)) if os.path.exists(settings_path) else {}
    data["export_dir"] = "/content/drive/MyDrive/Dubby"  # merge: keeps cookies_file and other settings
    json.dump(data, open(settings_path, "w"), indent=2)

try:
    server.terminate()
except NameError:
    pass
log = open("/content/dubby.log", "w")
server = subprocess.Popen([DUBBY, "serve", "--host", "0.0.0.0", "--port", "8765", "--no-open"], stdout=log, stderr=subprocess.STDOUT, cwd=REPO_DIR, env=os.environ.copy())
for _ in range(120):
    if server.poll() is not None:
        print(open("/content/dubby.log").read()[-4000:])
        raise RuntimeError("❌ Dubby exited during startup — see the log above.")
    try:
        urllib.request.urlopen("http://127.0.0.1:8765/api/settings", timeout=2)
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("❌ Dubby did not answer within 2 minutes — see /content/dubby.log")
print("✅ Dubby is running (pid", server.pid, ")")

if TUNNEL == "cloudflare":
    if not os.path.exists("/usr/local/bin/cloudflared"):
        run("curl -fsSL -o /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared")
    cf = subprocess.Popen(["cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8765"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in cf.stdout:
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
        if m:
            print("🐨 Open the studio →", m.group(0))
            break
else:
    from google.colab import output
    from google.colab.output import eval_js
    print("🐨 Open the studio →", eval_js("google.colab.kernel.proxyPort(8765)"))
    output.serve_kernel_port_as_window(8765, anchor_text="Open Dubby studio in a new tab")

### 📟 Live terminal (re-run any time)

In [ ]:
#@title Show the last studio log lines { display-mode: "form" }
LINES = 80  #@param {type:"integer"}
print("".join(open("/content/dubby.log", encoding="utf-8", errors="replace").readlines()[-LINES:]))

## 🤖 Optional: dub headlessly

The same pipeline without the UI. `source = auto` detects the spoken language, and every stage uses the **recommended engine** for the language unless you choose one.

In [ ]:
#@title Headless dub { display-mode: "form" }
URL = "https://www.youtube.com/watch?v=jNQXAC9IVRw"  #@param {type:"string"}
SOURCE = "auto"  #@param ["auto", "en", "ar", "es", "fr", "it", "hi", "zh", "ja"]
TARGET = "es"  #@param ["arz", "arb", "en", "es", "fr", "it", "hi", "zh", "ja"]
ASR = "recommended"  #@param ["recommended", "whisperx", "qwen3-asr", "cohere-transcribe", "cohere-transcribe-arabic", "coherex", "qwencleo", "parakeet", "metro-asr"]
TRANSLATION = "recommended"  #@param ["recommended", "hunyuan-mt", "nllb", "llm", "emhotob", "masrawy", "jisr", "indictrans2", "passthrough"]
TTS = "recommended"  #@param ["recommended", "omnivoice", "qwen3-tts", "voicetut", "lahgtna-omnivoice", "indicf5"]
VOICE = "auto"  #@param ["auto", "preset:Mohamed", "preset:Asmaa"]
opts = "".join(f" --{k} {v}" for k, v in (("asr", ASR), ("translation", TRANSLATION), ("tts", TTS)) if v != "recommended")
run(f"dubby dub '{URL}' --source {SOURCE} --target {TARGET}{opts} --voice {VOICE} --export /content/exports")

## 🛑 Stop the studio

In [ ]:
try:
    server.terminate()
    print("Stopped.")
except NameError:
    print("Server was not started from this notebook session.")